# 02 Train

Colab can only mount the **Google account you opened this notebook with**. The 84k-clip dataset lives on the original Drive folder:

https://drive.google.com/drive/folders/1MYLz3U9FfDxMAnqB97kMi01ZxgijxPnw?usp=sharing

**One-time, in the same Google account as this Colab:**

1. Open that link.
2. Right-click the folder → **Organize → Add shortcut → My Drive**.
3. Keep the name `smart_turn_en_hi_100k` if you can.
4. Re-run the next cell. It looks up folder id `1MYLz3U9FfDxMAnqB97kMi01ZxgijxPnw` and points training at it.

Do **not** rebuild the dataset. If the shortcut is missing, the cell will stop and tell you.

If the last run died with `No space left on device`, use **Runtime → Disconnect and delete runtime** first.

In [ ]:
from google.colab import drive
import os, shutil, subprocess, sys
from pathlib import Path

drive.mount("/content/drive")

SHARED_FOLDER_ID = "1MYLz3U9FfDxMAnqB97kMi01ZxgijxPnw"
SHARE_URL = f"https://drive.google.com/drive/folders/{SHARED_FOLDER_ID}?usp=sharing"


def is_hf_dataset(path: Path) -> bool:
    return path.is_dir() and (path / "dataset_info.json").exists()


def find_shared_subset() -> Path | None:
    candidates = [
        Path("/content/drive/MyDrive/smart_turn_en_hi_100k"),
        Path(f"/content/drive/.shortcut-targets-by-id/{SHARED_FOLDER_ID}"),
    ]
    shortcut = Path(f"/content/drive/.shortcut-targets-by-id/{SHARED_FOLDER_ID}")
    if shortcut.exists():
        candidates.extend(p for p in shortcut.iterdir() if p.is_dir())
    mydrive = Path("/content/drive/MyDrive")
    if mydrive.exists():
        for path in mydrive.iterdir():
            name = path.name.lower()
            if path.is_dir() and ("smart_turn" in name or "en_hi" in name):
                candidates.append(path)
    seen: set[Path] = set()
    for path in candidates:
        try:
            path = path.resolve()
        except OSError:
            continue
        if path in seen or not path.exists():
            continue
        seen.add(path)
        if is_hf_dataset(path):
            return path
        if path.is_dir():
            for child in path.iterdir():
                if child.is_dir() and is_hf_dataset(child):
                    return child
    return None


subset = find_shared_subset()
if subset is None:
    raise FileNotFoundError(
        "Shared dataset not visible to this Colab account yet.\n"
        f"1. Open this link while logged into the SAME Google account as Colab:\n   {SHARE_URL}\n"
        "2. Right-click the folder → Organize → Add shortcut → My Drive.\n"
        "3. Re-run this cell. Do not rebuild the 84k clips."
    )
print(f"using shared dataset at {subset}")

for cache in [Path.home() / ".cache/huggingface", Path("/root/.cache/huggingface")]:
    if cache.exists():
        shutil.rmtree(cache, ignore_errors=True)

repo = Path("/content/ShipRocket-assesment")
if repo.exists():
    subprocess.check_call(["git", "-C", str(repo), "pull"])
else:
    subprocess.check_call(
        ["git", "clone", "https://github.com/Saaalil/ShipRocket-assesment.git", str(repo)]
    )
os.chdir(repo)
for yaml_path in [Path("configs/partial_unfreeze.yaml")]:
    lines = []
    for line in yaml_path.read_text().splitlines(True):
        if line.startswith("subset_cache:"):
            lines.append(f"subset_cache: {subset}\n")
        else:
            lines.append(line)
    yaml_path.write_text("".join(lines))
print("configs now point at the shared Drive folder")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"])

In [ ]:
# Final training run (partial unfreeze).
!python scripts/train.py --config configs/partial_unfreeze.yaml

In [ ]:
# 1 epoch, save every 100 steps, eval only at the end (~2–3h on T4).
# If Colab drops, re-run this cell; it resumes from the latest Drive checkpoint.
!python scripts/train.py --config configs/partial_unfreeze.yaml